<a href="https://colab.research.google.com/github/Anjana71/ai_agent/blob/main/ai_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install gradio
from datetime import datetime, timedelta
import gradio as gr

# Store all academic info
academic_data = {
    'exams': [],
    'tasks': [],
    'teachers': {}
}


In [16]:
# Safe date parser
def safe_parse_date(date_str):
    try:
        return datetime.strptime(date_str.strip(), "%Y-%m-%d")
    except Exception:
        return None

def add_exam(subject, date_str):
    date = safe_parse_date(date_str)
    if not date:
        return "❌ Invalid exam date format. Use YYYY-MM-DD."
    academic_data['exams'].append({'subject': subject, 'date': date})
    return f"✅ Exam '{subject}' on {date.date()} added!"

def add_task(description, due_date_str):
    due = safe_parse_date(due_date_str)
    if not due:
        return "❌ Invalid task date format. Use YYYY-MM-DD."
    academic_data['tasks'].append({'description': description, 'due_date': due})
    return f"📝 Task '{description}' due on {due.date()} added!"

def add_teacher(name, subject, traits):
    academic_data['teachers'][name] = {'subject': subject, 'traits': traits}
    return f"👩‍🏫 Teacher '{name}' for {subject} added!"


In [17]:
def daily_brief():
    today = datetime.now().date()
    response = ""

    for exam in academic_data['exams']:
        days_left = (exam['date'].date() - today).days
        if days_left == 0:
            response += f"📘 Exam today: {exam['subject']}\n"
        elif days_left == 1:
            response += f"⚠️ {exam['subject']} exam is tomorrow!\n"
        elif 1 < days_left <= 5:
            response += f"📚 {exam['subject']} exam in {days_left} days.\n"

    for task in academic_data['tasks']:
        t_days = (task['due_date'].date() - today).days
        if t_days == 0:
            response += f"🔔 Task due today: {task['description']}\n"
        elif 0 < t_days <= 2:
            response += f"🕒 Task coming: {task['description']} in {t_days} days.\n"

    return response if response else "📭 No upcoming deadlines."

def friendly_chat(message):
    msg = message.lower()
    if "tired" in msg:
        return "😌 Take a short break. Then we’ll do a quick revision."
    elif "exam" in msg or "task" in msg:
        return "🔍 Let me check your schedule...\n" + daily_brief()
    elif "hello" in msg or "hi" in msg:
        return "Hey friend! How can I help you with your studies today?"
    else:
        return "📚 Let's stay focused. Need a revision reminder?"


In [19]:
def add_data_tab(exam_subj, exam_date, task_desc, task_due, teacher_name, teacher_subject, traits):
    messages = []

    # Exam input
    if exam_subj.strip() and exam_date.strip():
        messages.append(add_exam(exam_subj, exam_date))
    elif exam_subj.strip() or exam_date.strip():
        messages.append("⚠️ To add an exam, both subject and date are needed.")

    # Task input
    if task_desc.strip() and task_due.strip():
        messages.append(add_task(task_desc, task_due))
    elif task_desc.strip() or task_due.strip():
        messages.append("⚠️ To add a task, both description and due date are needed.")

    # Teacher input
    if teacher_name.strip() and teacher_subject.strip():
        messages.append(add_teacher(teacher_name, teacher_subject, traits))
    elif teacher_name.strip() or teacher_subject.strip():
        messages.append("⚠️ To add a teacher, name and subject are needed.")

    if not messages:
        return "⚠️ No valid input provided. Please fill at least one full set of fields."

    reminders = daily_brief()
    return "\n".join(messages) + "\n\n📢 Current Reminders:\n" + reminders


In [20]:
with gr.Blocks() as demo:
    gr.Markdown("## 📘 AI Academic Assistant - Study Buddy")

    with gr.Tab("➕ Add Info"):
        with gr.Row():
            exam_subj = gr.Textbox(label="Exam Subject")
            exam_date = gr.Textbox(label="Exam Date (YYYY-MM-DD)")
        with gr.Row():
            task_desc = gr.Textbox(label="Task Description")
            task_due = gr.Textbox(label="Task Due Date (YYYY-MM-DD)")
        with gr.Row():
            teacher_name = gr.Textbox(label="Teacher Name")
            teacher_subject = gr.Textbox(label="Subject")
            traits = gr.Textbox(label="Traits (e.g., strict, conceptual)")
        submit = gr.Button("Add Information")
        output = gr.Textbox(label="Assistant Response")
        submit.click(
            add_data_tab,
            inputs=[exam_subj, exam_date, task_desc, task_due, teacher_name, teacher_subject, traits],
            outputs=output
        )

    with gr.Tab("💬 Chat with Assistant"):
        chat_input = gr.Textbox(label="Type a message...")
        chat_response = gr.Textbox(label="AI Assistant Reply")
        chat_input.submit(friendly_chat, inputs=chat_input, outputs=chat_response)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0e9185c9feafab291a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
